In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("/workspace/data/cotacoes_preprocessadas.sqlite")

# 1. Intervalo de datas e sugestão de data limite de treino (80%)
datas = pd.read_sql_query("""
    SELECT 
        MIN(data_cotacao) AS data_inicial,
        MAX(data_cotacao) AS data_final,
        COUNT(DISTINCT data_cotacao) AS total_datas
    FROM cotacoes_convertidas_kg
    WHERE outlier_suspeito = 0
""", conn)
print("=== INTERVALO DE DATAS (R$/kg sem outliers) ===")
display(datas)

lista_datas = pd.read_sql_query("""
    SELECT DISTINCT data_cotacao 
    FROM cotacoes_convertidas_kg 
    WHERE outlier_suspeito = 0 
    ORDER BY data_cotacao
""", conn)["data_cotacao"].tolist()
print(f"Sugestão de CEASA_DATA_LIMITE_INFERENCIA (80% das datas): {lista_datas[int(len(lista_datas) * 0.8)]}\n")

# 2. Top 15 produtos com preço em R$/kg em mais entrepostos e datas
top_produtos = pd.read_sql_query("""
    SELECT 
        produto_chave_textual AS produto,
        COUNT(*) AS total_cotacoes_kg,
        COUNT(DISTINCT entreposto_slug) AS qtd_entrepostos,
        COUNT(DISTINCT ceasa_slug) AS qtd_ceasas,
        COUNT(DISTINCT data_cotacao) AS qtd_datas,
        MIN(data_cotacao) AS inicio,
        MAX(data_cotacao) AS fim
    FROM cotacoes_convertidas_kg
    WHERE outlier_suspeito = 0
    GROUP BY produto_chave_textual
    HAVING qtd_entrepostos >= 3
    ORDER BY qtd_datas DESC, qtd_entrepostos DESC
    LIMIT 15
""", conn)
print("=== TOP 15 PRODUTOS PARA SÉRIES TEMPORAIS (R$/kg) ===")
display(top_produtos)

# 3. Top regras inferidas aguardando data de corte
regras = pd.read_sql_query("""
    SELECT 
        produto_chave_textual,
        assinatura_inferencia,
        peso_total_kg_inferido,
        suporte_candidato,
        suporte_ancora,
        motivo_status
    FROM regras_conversao_inferida
    WHERE motivo_status = 'data_limite_treino_nao_definida'
    ORDER BY suporte_candidato DESC
    LIMIT 15
""", conn)
print("=== TOP REGRAS INFERIDAS PRONTAS PARA ATIVAR ===")
display(regras)

=== INTERVALO DE DATAS (R$/kg sem outliers) ===


,data_inicial,data_final,total_datas
0,2010-09-01,2026-11-11,2550


Sugestão de CEASA_DATA_LIMITE_INFERENCIA (80% das datas): 2024-11-26

=== TOP 15 PRODUTOS PARA SÉRIES TEMPORAIS (R$/kg) ===


,produto,total_cotacoes_kg,qtd_entrepostos,qtd_ceasas,qtd_datas,inicio,fim
0,morango,13402,15,4,2492,2010-09-03,2026-11-11
1,couve,15502,11,4,2488,2010-09-01,2026-11-11
2,salsa,4442,5,3,2443,2010-09-01,2026-11-10
3,hortela,8038,9,4,2440,2015-07-02,2026-11-10
4,cebolinha,7337,10,4,2424,2010-09-01,2026-11-10
5,rabanete,7425,10,4,2401,2010-09-01,2026-11-11
6,maxixe,5829,11,5,2385,2010-09-05,2026-11-10
7,batata doce,18037,15,5,2383,2016-10-03,2026-11-11
8,graviola,4578,13,6,2378,2015-07-02,2026-11-10
9,quiabo,14341,15,5,2374,2016-10-03,2026-11-11


=== TOP REGRAS INFERIDAS PRONTAS PARA ATIVAR ===


,produto_chave_textual,assinatura_inferencia,peso_total_kg_inferido,suporte_candidato,suporte_ancora,motivo_status
0,maxixe,caixa 18,18.0,542,9,data_limite_treino_nao_definida
1,pimentao,caixa 10,10.0,539,197,data_limite_treino_nao_definida
2,beterraba,caixa 20,20.0,523,34,data_limite_treino_nao_definida
3,goiaba,caixa 20,20.0,271,3088,data_limite_treino_nao_definida
4,graviola,caixa 10,10.0,271,45,data_limite_treino_nao_definida
5,kiwi nacional,caixa 10,10.0,271,161,data_limite_treino_nao_definida
6,limao galego,caixa 20,20.0,270,78,data_limite_treino_nao_definida
7,pinha,caixa 3,3.0,270,2547,data_limite_treino_nao_definida
8,morango,caixa 1 5,1.5,268,297,data_limite_treino_nao_definida
9,roma,caixa 4,4.0,268,1947,data_limite_treino_nao_definida


In [3]:
# Célula 2: Encontrar os melhores pares (Produto + Entreposto A vs Entreposto B de outra CEASA)
# com mais datas em comum para o Teste de Granger

conn.executescript("""
DROP TABLE IF EXISTS temp.series_diarias;

-- 1. Consolida 1 preço mediano (R$/kg) por produto, entreposto e dia (sem outliers)
CREATE TEMP TABLE series_diarias AS
SELECT
    produto_chave_textual AS produto,
    ceasa_slug,
    entreposto_slug,
    data_cotacao,
    AVG(preco_comum_kg) AS preco_kg
FROM cotacoes_convertidas_kg
WHERE outlier_suspeito = 0
GROUP BY produto_chave_textual, ceasa_slug, entreposto_slug, data_cotacao;

CREATE INDEX temp.idx_series_diarias_prod_data 
    ON series_diarias(produto, data_cotacao);
""")

# 2. Pares de entrepostos de CEASAs diferentes com mais datas exatamente no mesmo dia
pares_diarios = pd.read_sql_query("""
    WITH series_fortes AS (
        SELECT produto, ceasa_slug, entreposto_slug, COUNT(*) AS total_dias
        FROM temp.series_diarias
        GROUP BY produto, ceasa_slug, entreposto_slug
        HAVING COUNT(*) >= 300
    )
    SELECT
        a.produto,
        a.entreposto_slug AS mercado_A,
        b.entreposto_slug AS mercado_B,
        COUNT(*) AS datas_em_comum,
        MIN(a.data_cotacao) AS inicio_comum,
        MAX(a.data_cotacao) AS fim_comum
    FROM temp.series_diarias a
    JOIN temp.series_diarias b
      ON a.produto = b.produto
     AND a.data_cotacao = b.data_cotacao
     AND a.ceasa_slug < b.ceasa_slug
    JOIN series_fortes sa
      ON sa.produto = a.produto AND sa.entreposto_slug = a.entreposto_slug
    JOIN series_fortes sb
      ON sb.produto = b.produto AND sb.entreposto_slug = b.entreposto_slug
    GROUP BY a.produto, a.entreposto_slug, b.entreposto_slug
    ORDER BY datas_em_comum DESC
    LIMIT 25
""", conn)

print("=== TOP 25 PARES CROSS-CEASA (DATAS EXATAMENTE EM COMUM - DIÁRIO) ===")
display(pares_diarios)

# 3. E os principais produtos agrícolas (tomate, batata, cebola, cenoura, pimentao, banana)?
principais = pd.read_sql_query("""
    SELECT 
        produto,
        COUNT(DISTINCT ceasa_slug) AS qtd_ceasas,
        COUNT(DISTINCT entreposto_slug) AS qtd_entrepostos,
        COUNT(DISTINCT data_cotacao) AS qtd_datas,
        COUNT(*) AS pontos_serie_diaria,
        MIN(data_cotacao) AS inicio,
        MAX(data_cotacao) AS fim
    FROM temp.series_diarias
    WHERE produto LIKE '%tomate%'
       OR produto LIKE '%batata%'
       OR produto LIKE '%cebola%'
       OR produto LIKE '%cenoura%'
       OR produto LIKE '%pimentao%'
       OR produto LIKE '%banana%'
       OR produto LIKE '%alface%'
       OR produto LIKE '%morango%'
       OR produto LIKE '%melao%'
    GROUP BY produto
    HAVING qtd_ceasas >= 3
    ORDER BY pontos_serie_diaria DESC
    LIMIT 25
""", conn)

print("=== VARIEDADES DOS PRINCIPAIS PRODUTOS (EM >= 3 CEASAS) ===")
display(principais)

=== TOP 25 PARES CROSS-CEASA (DATAS EXATAMENTE EM COMUM - DIÁRIO) ===


,produto,mercado_A,mercado_B,datas_em_comum,inicio_comum,fim_comum
0,couve,ceasa-grande-vitoria,recife,1184,2021-08-02,2026-11-10
1,salsa,ceasa-grande-vitoria,recife,1179,2021-08-02,2026-11-10
2,hortela,ceasa-grande-vitoria,recife,1164,2021-08-02,2026-11-10
3,beterraba,recife,curitiba,1101,2022-01-04,2026-11-10
4,couve,recife,curitiba,1101,2022-01-04,2026-11-10
5,hortela,recife,curitiba,1101,2022-01-04,2026-11-10
6,quiabo,recife,curitiba,1101,2022-01-04,2026-11-10
7,milho,recife,curitiba,1100,2022-01-04,2026-11-10
8,feijao,recife,curitiba,1099,2022-01-04,2026-11-10
9,maxixe,recife,curitiba,1098,2022-01-04,2026-11-10


=== VARIEDADES DOS PRINCIPAIS PRODUTOS (EM >= 3 CEASAS) ===


,produto,qtd_ceasas,qtd_entrepostos,qtd_datas,pontos_serie_diaria,inicio,fim
0,morango,4,15,2492,8638,2010-09-03,2026-11-11
1,cenoura,6,18,2033,7811,2016-10-03,2026-11-11
2,batata doce,5,15,2383,7484,2016-10-03,2026-11-11
3,cebola,5,15,2359,7182,2017-05-15,2026-11-11
4,batata,5,14,1225,6745,2022-01-04,2026-11-11
5,melao,6,16,2361,6380,2017-05-15,2026-11-11
6,tomate,4,14,1299,5686,2019-10-21,2026-11-11
7,banana prata,5,14,1477,4910,2016-10-03,2026-11-09
8,pimentao verde,5,9,1843,4895,2019-05-29,2026-11-10
9,pimentao,5,17,1853,4821,2016-10-03,2026-11-11


In [4]:
import numpy as np

# 1. Top pares Cross-CEASA apenas para as grandes commodities agrícolas
commodities = (
    'cenoura', 'cebola', 'batata', 'batata doce', 
    'tomate', 'morango', 'beterraba', 'melao', 
    'banana prata', 'pimentao', 'pimentao verde', 'quiabo'
)

pares_commodities = pd.read_sql_query(f"""
    SELECT
        a.produto,
        a.ceasa_slug AS ceasa_A,
        a.entreposto_slug AS mercado_A,
        b.ceasa_slug AS ceasa_B,
        b.entreposto_slug AS mercado_B,
        COUNT(*) AS datas_em_comum,
        MIN(a.data_cotacao) AS inicio_comum,
        MAX(a.data_cotacao) AS fim_comum
    FROM temp.series_diarias a
    JOIN temp.series_diarias b
      ON a.produto = b.produto
     AND a.data_cotacao = b.data_cotacao
     AND a.ceasa_slug < b.ceasa_slug
    WHERE a.produto IN {commodities}
      AND a.data_cotacao >= '2022-01-01'
    GROUP BY a.produto, a.entreposto_slug, b.entreposto_slug
    HAVING datas_em_comum >= 500
    ORDER BY datas_em_comum DESC
    LIMIT 20
""", conn)

print("=== TOP 20 PARES CROSS-CEASA (GRANDES COMMODITIES A PARTIR DE 2022) ===")
display(pares_commodities)

# 2. Função simples de Causalidade de Granger (OLS F-test em retornos logarítmicos)
# Testa se mudanças passadas no Mercado X ajudam a prever mudanças no Mercado Y
def testar_granger_par(produto, mercado_x, mercado_y, max_lag=5, ffill_limit=3):
    df = pd.read_sql_query("""
        SELECT data_cotacao, entreposto_slug, preco_kg
        FROM temp.series_diarias
        WHERE produto = ?
          AND entreposto_slug IN (?, ?)
          AND data_cotacao >= '2022-01-01'
        ORDER BY data_cotacao
    """, conn, params=(produto, mercado_x, mercado_y))
    
    piv = df.pivot(index="data_cotacao", columns="entreposto_slug", values="preco_kg")
    piv.index = pd.to_datetime(piv.index)
    # Calendário de dias úteis com preenchimento de até 3 dias úteis (feriados/falhas curtas)
    piv = piv.asfreq("B").ffill(limit=ffill_limit)
    
    # Retorno logarítmico (diferença do log) para estacionariedade
    ret = np.log(piv).diff().dropna()
    if len(ret) < 100:
        return None
    
    y = ret[mercado_y].values
    x = ret[mercado_x].values
    
    # Monta matriz defasada para lag = max_lag
    n = len(y) - max_lag
    Y = y[max_lag:]
    lags_y = np.column_stack([y[max_lag - k : len(y) - k] for k in range(1, max_lag + 1)])
    lags_x = np.column_stack([x[max_lag - k : len(x) - k] for k in range(1, max_lag + 1)])
    
    X_restrito = np.column_stack([np.ones(n), lags_y])
    X_irrestrito = np.column_stack([np.ones(n), lags_y, lags_x])
    
    rss_r = np.sum((Y - X_restrito @ np.linalg.lstsq(X_restrito, Y, rcond=None)[0]) ** 2)
    rss_u = np.sum((Y - X_irrestrito @ np.linalg.lstsq(X_irrestrito, Y, rcond=None)[0]) ** 2)
    
    df1 = max_lag
    df2 = n - X_irrestrito.shape[1]
    f_stat = ((rss_r - rss_u) / df1) / (rss_u / df2)
    return {"n_obs": n, "F_stat": round(float(f_stat), 3), "critico_5pct_aprox": 2.22}

# Testa os 8 primeiros pares nas duas direções (A -> B e B -> A)
resultados_granger = []
for row in pares_commodities.head(8).itertuples():
    ab = testar_granger_par(row.produto, row.mercado_A, row.mercado_B, max_lag=5)
    ba = testar_granger_par(row.produto, row.mercado_B, row.mercado_A, max_lag=5)
    if ab and ba:
        resultados_granger.append({
            "produto": row.produto,
            "par": f"{row.mercado_A} <-> {row.mercado_B}",
            "n_dias_uteis": ab["n_obs"],
            "F (A -> B)": ab["F_stat"],
            "A causa B? (F>2.22)": "SIM" if ab["F_stat"] > 2.22 else "nao",
            "F (B -> A)": ba["F_stat"],
            "B causa A? (F>2.22)": "SIM" if ba["F_stat"] > 2.22 else "nao",
        })

print("=== TESTE PRELIMINAR DE GRANGER (RETORNOS LOG, LAG=5 DIAS ÚTEIS) ===")
display(pd.DataFrame(resultados_granger))

=== TOP 20 PARES CROSS-CEASA (GRANDES COMMODITIES A PARTIR DE 2022) ===


,produto,ceasa_A,mercado_A,ceasa_B,mercado_B,datas_em_comum,inicio_comum,fim_comum
0,beterraba,ceasa-pe,recife,ceasa-pr,curitiba,1101,2022-01-04,2026-11-10
1,quiabo,ceasa-pe,recife,ceasa-pr,curitiba,1101,2022-01-04,2026-11-10
2,morango,ceasa-pe,recife,ceasa-pr,curitiba,1096,2022-01-04,2026-11-10
3,cebola,ceasa-pe,recife,ceasa-pr,maringa,1076,2022-01-11,2026-11-10
4,quiabo,ceasa-pe,recife,ceasa-pr,maringa,1076,2022-01-11,2026-11-10
5,melao,ceasa-pe,recife,ceasa-pr,maringa,1069,2022-01-11,2026-11-10
6,morango,ceasa-pe,recife,ceasa-pr,cascavel,1068,2022-01-11,2026-11-10
7,quiabo,ceasa-pe,recife,ceasa-pr,cascavel,1068,2022-01-11,2026-11-10
8,morango,ceasa-pe,recife,ceasa-pr,maringa,1061,2022-01-11,2026-11-10
9,quiabo,ceasa-pe,recife,ceasa-pr,foz-do-iguacu,1061,2022-01-11,2026-11-09


=== TESTE PRELIMINAR DE GRANGER (RETORNOS LOG, LAG=5 DIAS ÚTEIS) ===


,produto,par,n_dias_uteis,F (A -> B),A causa B? (F>2.22),F (B -> A),B causa A? (F>2.22)
0,beterraba,recife <-> curitiba,1208,0.643,nao,1.983,nao
1,quiabo,recife <-> curitiba,1208,0.399,nao,1.337,nao
2,morango,recife <-> curitiba,1205,0.404,nao,3.186,SIM
3,cebola,recife <-> maringa,1203,8.941,SIM,2.734,SIM
4,quiabo,recife <-> maringa,1203,2.341,SIM,2.668,SIM
5,melao,recife <-> maringa,1201,2.409,SIM,0.469,nao
6,morango,recife <-> cascavel,1202,6.888,SIM,1.746,nao
7,quiabo,recife <-> cascavel,1202,1.143,nao,0.850,nao


In [6]:
# Célula 4: Visão geral de todos os Entrepostos/CEASAs no banco e teste intra-estado (ex: PR)
visao_mercados = pd.read_sql_query("""
    SELECT 
        ceasa_slug,
        uf,
        entreposto_slug,
        COUNT(*) AS cotacoes_kg,
        COUNT(DISTINCT produto_chave_textual) AS qtd_produtos,
        COUNT(DISTINCT data_cotacao) AS qtd_datas,
        MIN(data_cotacao) AS inicio,
        MAX(data_cotacao) AS fim
    FROM cotacoes_convertidas_kg
    WHERE outlier_suspeito = 0
    GROUP BY ceasa_slug, uf, entreposto_slug
    ORDER BY qtd_datas DESC
""", conn)

print("=== TODOS OS ENTREPOSTOS DISPONÍVEIS COM PREÇO EM R$/KG ===")
display(visao_mercados)

# Teste rápido entre mercados do MESMO estado (ex: Curitiba <-> Maringá no PR)
intra_pr = []
for prod in ['tomate', 'batata', 'cebola', 'cenoura', 'morango', 'beterraba', 'pimentao verde', 'banana prata']:
    ab = testar_granger_par(prod, 'curitiba', 'maringa', max_lag=5)
    ba = testar_granger_par(prod, 'maringa', 'curitiba', max_lag=5)
    if ab and ba:
        intra_pr.append({
            "produto": prod,
            "par": "curitiba <-> maringa (CEASA-PR)",
            "n_dias_uteis": ab["n_obs"],
            "F (Curitiba -> Maringa)": ab["F_stat"],
            "Curitiba -> Maringa?": "SIM" if ab["F_stat"] > 2.22 else "nao",
            "F (Maringa -> Curitiba)": ba["F_stat"],
            "Maringa -> Curitiba?": "SIM" if ba["F_stat"] > 2.22 else "nao",
        })

print("=== TESTE DE GRANGER INTRA-ESTADO (CURITIBA <-> MARINGÁ, LAG=5) ===")
display(pd.DataFrame(intra_pr))

=== TODOS OS ENTREPOSTOS DISPONÍVEIS COM PREÇO EM R$/KG ===


,ceasa_slug,uf,entreposto_slug,cotacoes_kg,qtd_produtos,qtd_datas,inicio,fim
0,ceasa-pe,PE,recife,297033,163,2269,2017-05-15,2026-11-10
1,ceasa-es,ES,ceasa-grande-vitoria,184413,417,1272,2021-08-02,2026-11-10
2,ceasa-pr,PR,curitiba,433285,380,1152,2022-01-04,2026-11-10
3,ceasa-pr,PR,maringa,189136,304,1125,2022-01-11,2026-11-10
4,ceasa-pr,PR,cascavel,155504,299,1123,2022-01-11,2026-11-11
5,ceasa-pr,PR,foz-do-iguacu,219343,414,1123,2022-01-11,2026-11-09
6,ceasa-pr,PR,londrina,252736,317,1065,2022-01-11,2026-11-09
7,ceasa-es,ES,ceasa-noroeste,33683,173,850,2010-09-01,2024-12-20
8,ceasa-go,GO,goiania,7227,48,300,2025-03-04,2026-11-10
9,ceasa-es,ES,ceasa-cachoeiro-de-itapemirim,19183,196,180,2015-07-02,2018-08-27


=== TESTE DE GRANGER INTRA-ESTADO (CURITIBA <-> MARINGÁ, LAG=5) ===


,produto,par,n_dias_uteis,F (Curitiba -> Maringa),Curitiba -> Maringa?,F (Maringa -> Curitiba),Maringa -> Curitiba?
0,tomate,curitiba <-> maringa (CEASA-PR),929,6.492,SIM,16.042,SIM
1,batata,curitiba <-> maringa (CEASA-PR),1215,18.051,SIM,4.398,SIM
2,cebola,curitiba <-> maringa (CEASA-PR),1120,11.688,SIM,2.552,SIM
3,cenoura,curitiba <-> maringa (CEASA-PR),1196,6.468,SIM,7.915,SIM
4,morango,curitiba <-> maringa (CEASA-PR),1203,9.534,SIM,11.278,SIM
5,beterraba,curitiba <-> maringa (CEASA-PR),529,1.563,nao,0.081,nao
6,pimentao verde,curitiba <-> maringa (CEASA-PR),527,4.964,SIM,0.714,nao
7,banana prata,curitiba <-> maringa (CEASA-PR),527,3.070,SIM,1.459,nao
